# Data Preprocessing Pipeline

**Dataset:** GoodScents + Leffingwell 

**Feature families:**
- MACCS keys: 166 binary bits
- Morgan fingerprints: 512 binary bits (radius=2)
- Mordred descriptors: 327 continuous physicochemical descriptors 

## Imports

In [27]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from skmultilearn.model_selection import iterative_train_test_split


## Load Data

In [28]:
df = pd.read_csv(r'C:\Users\hp\Desktop\stage etis\scripts\dataset_A_radius_1.csv', sep=';')

LABEL_COLS = [
    'floral', 'fruity', 'sweet', 'woody', 'green', 'spicy',
    'animal_musk', 'earthy', 'citrus', 'chemical', 'gourmand', 'powdery_amber'
]

#Kanfr9o fingerprints 3la mordred descriptors 7it antraitiwhoum in different ways 

fp_cols      = [c for c in df.columns if c.startswith('MACCS_') or c.startswith('morgan_')]
mordred_cols = [c for c in df.columns if c not in LABEL_COLS + ['SMILES'] + fp_cols]

print(f'Total molecules      : {len(df)}')
print(f'MACCS + Morgan cols  : {len(fp_cols)}')
print(f'Mordred cols         : {len(mordred_cols)}')
print(f'Label cols           : {len(LABEL_COLS)}')

Total molecules      : 4981
MACCS + Morgan cols  : 678
Mordred cols         : 327
Label cols           : 12


## Inspect NaN Values

The goal is to remove all  the columns where there's at least one  NAN value but here  when Inspected the the df i found that the the rows that has nan values are 5 for 327 mordred which is the mordred descriptors so instead of dropping the colums i dropped the rows 

In [29]:
X_all = df[fp_cols + mordred_cols].values.astype(float)

#X7al mn row ou column fiha NaNs kan7sbouha bsum dyal boolean mask li kaydir True ila kayn NaN, w False ila ma kaynch, w kan7sbouha bsum bash n3rfou ch7al mn row ou column fiha NaNs

nan_rows = np.isnan(X_all).any(axis=1).sum()
nan_cols = np.isnan(X_all).any(axis=0).sum()

nan_cols_fp      = np.isnan(df[fp_cols].values.astype(float)).any(axis=0).sum()
nan_cols_mordred = np.isnan(df[mordred_cols].values.astype(float)).any(axis=0).sum()

nan_row_indices = list(np.where(np.isnan(X_all).any(axis=1))[0])

print(f'Rows with NaN        : {nan_rows}')
print(f'Columns with NaN     : {nan_cols}')
print(f'  - in fingerprints  : {nan_cols_fp}')
print(f'  - in Mordred       : {nan_cols_mordred}')

print()
print('SMILES of problematic molecules:')
for idx in nan_row_indices:
    print(f'  row {idx}: {df.iloc[idx]["SMILES"]}')


Rows with NaN        : 5
Columns with NaN     : 327
  - in fingerprints  : 0
  - in Mordred       : 327

SMILES of problematic molecules:
  row 3917: CC(C)=CCCC(C)CC(OCCCCCCCCCCC(C)C)OCCCCCCCCCCC(C)C
  row 3943: CC(C)CCCCCCCCCCOC(CC(C)CCCC(C)(C)O)OCCCCCCCCCCC(C)C
  row 3977: CC1=C2CC(C=O)C(C1)C(C(C)C)C2
  row 3978: CC(=O)C1CC2=C(C)CC1C(C(C)C)C2
  row 4646: CC(C)CCCCCCCCCCCCCCC(=O)OCC(O)COCC(COC(=O)CCCCCCCCCCCCCCC(C)C)OC(=O)CCCCCCCCCCCCCCC(C)C


## Drop NaN Rows


In [30]:
df_clean = df.dropna().reset_index(drop=True)


print(f'Molecules before : {len(df)}')
print(f'Molecules after  : {len(df_clean)}')
print(f'Dropped          : {len(df) - len(df_clean)}')
print(f'NaN remaining    : {df_clean[fp_cols + mordred_cols].isna().sum().sum()}')

Molecules before : 4981
Molecules after  : 4976
Dropped          : 5
NaN remaining    : 0


## Split into Feature Groups

In [31]:
X_fp      = df_clean[fp_cols].values.astype(float)
X_mordred = df_clean[mordred_cols].values.astype(float)
y         = df_clean[LABEL_COLS].values

print(f'X_fingerprints shape : {X_fp.shape}')
print(f'X_mordred shape      : {X_mordred.shape}')
print(f'y shape              : {y.shape}')

X_fingerprints shape : (4976, 678)
X_mordred shape      : (4976, 327)
y shape              : (4976, 12)


## Remove Zero-Variance Features

Features with zero variance are identical across all molecules  so , they carry no information and should be removed.
This is applied independently to each feature group.


In [32]:
# Fingerprints
vt_fp = VarianceThreshold(threshold=0)
#kayselecti lcolumns li 3andhom variance > 0, w kaydroppihoum li 3andhom variance = 0 
X_fp = vt_fp.fit_transform(X_fp)
fp_cols_kept = np.array(fp_cols)[vt_fp.get_support()]

print(f'Fingerprints: {len(fp_cols)} -> {X_fp.shape[1]} features '
      f'({len(fp_cols) - X_fp.shape[1]} zero-variance dropped)')

# Mordred
vt_mordred = VarianceThreshold(threshold=0)
X_mordred = vt_mordred.fit_transform(X_mordred)
mordred_cols_kept = np.array(mordred_cols)[vt_mordred.get_support()]

print(f'Mordred     : {len(mordred_cols)} -> {X_mordred.shape[1]} features '
      f'({len(mordred_cols) - X_mordred.shape[1]} zero-variance dropped)')
print(f'\nTotal features after zero-variance removal: {X_fp.shape[1] + X_mordred.shape[1]}')

Fingerprints: 678 -> 647 features (31 zero-variance dropped)
Mordred     : 327 -> 327 features (0 zero-variance dropped)

Total features after zero-variance removal: 974


## Train/Test Split

I used iterative_train_test_split from scikit-multilearn cuz  For multi-label data with imbalanced labels, this  preserves the positive/negative ratio of each label independently in both train and test sets — critical given our class imbalance.

In [33]:

# I concatenated  temporarily just for the split, then separate again
X_combined = np.hstack([X_fp, X_mordred])
n_fp = X_fp.shape[1]

X_train_comb, y_train, X_test_comb, y_test = iterative_train_test_split(
    X_combined, y, test_size=0.2
)

# Separate back into fingerprints and Mordred
X_fp_train    = X_train_comb[:, :n_fp]
X_mordred_train = X_train_comb[:, n_fp:]
X_fp_test     = X_test_comb[:, :n_fp]
X_mordred_test  = X_test_comb[:, n_fp:]

print(f'Train : {X_fp_train.shape[0]} molecules')
print(f'Test  : {X_fp_test.shape[0]} molecules')
print()

print(f'{"Label":<20} {"Full":>8} {"Train":>8} {"Test":>8}')
print('-' * 48)
for i, l in enumerate(LABEL_COLS):
    full  = y[:, i].mean() * 100
    train = y_train[:, i].mean() * 100
    test  = y_test[:, i].mean() * 100
    print(f'{l:<20} {full:>7.1f}% {train:>7.1f}% {test:>7.1f}%')

Train : 3954 molecules
Test  : 1022 molecules

Label                    Full    Train     Test
------------------------------------------------
floral                  14.6%    14.7%    14.2%
fruity                  27.8%    28.0%    27.1%
sweet                   18.3%    18.5%    17.9%
woody                   13.2%    13.3%    12.8%
green                   31.2%    31.4%    30.3%
spicy                   16.1%    16.2%    15.7%
animal_musk             15.5%    15.6%    15.1%
earthy                  16.2%    16.3%    15.9%
citrus                   4.0%     3.9%     4.3%
chemical                43.1%    43.4%    42.0%
gourmand                24.2%    24.4%    23.6%
powdery_amber           20.6%    20.7%    20.1%


## Scale Mordred Features



In [34]:
scaler = StandardScaler()

# Fit on train, transform both
X_mordred_train = scaler.fit_transform(X_mordred_train)
X_mordred_test  = scaler.transform(X_mordred_test)      # same scaler, no refit

print('StandardScaler fitted on train only.')
print(f'Mordred train mean : {X_mordred_train.mean():.4f}')
print(f'Mordred train std  : {X_mordred_train.std():.4f}')
print(f'Mordred test mean  : {X_mordred_test.mean():.4f}')

StandardScaler fitted on train only.
Mordred train mean : -0.0000
Mordred train std  : 0.9985
Mordred test mean  : -0.0387


## Correlation Filter on Mordred (Train Only)

Highly correlated Mordred features (r > 0.95) are redundant ,  they encode the same information.
For each correlated pair, we drop one feature.

The correlation structure is computed on train only and the same column mask is applied to test.

In [35]:
# Compute correlation matrix on train
corr_matrix = np.corrcoef(X_mordred_train.T)
corr_matrix = np.abs(corr_matrix)

# Find columns to drop
upper_triangle = np.triu(corr_matrix, k=1)
cols_to_drop = set()
rows, cols = np.where(upper_triangle > 0.95)
for r, c in zip(rows, cols):
    if c not in cols_to_drop:
        cols_to_drop.add(c)

cols_to_keep = [i for i in range(X_mordred_train.shape[1]) if i not in cols_to_drop]

# Apply same mask to both train and test
X_mordred_train = X_mordred_train[:, cols_to_keep]
X_mordred_test  = X_mordred_test[:, cols_to_keep]

print(f'Mordred features before correlation filter : {len(cols_to_keep) + len(cols_to_drop)}')
print(f'Features dropped (r > 0.95)               : {len(cols_to_drop)}')
print(f'Mordred features remaining                : {X_mordred_train.shape[1]}')

Mordred features before correlation filter : 327
Features dropped (r > 0.95)               : 4
Mordred features remaining                : 323


C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\numpy\lib\function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\numpy\lib\function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


## Get the final feature Matrices



In [36]:
X_train = np.hstack([X_fp_train, X_mordred_train])
X_test  = np.hstack([X_fp_test,  X_mordred_test])

print('Final feature matrices:')
print(f'  X_train : {X_train.shape}')
print(f'  X_test  : {X_test.shape}')
print(f'  y_train : {y_train.shape}')
print(f'  y_test  : {y_test.shape}')
print()


Final feature matrices:
  X_train : (3954, 970)
  X_test  : (1022, 970)
  y_train : (3954, 12)
  y_test  : (1022, 12)




# Model: Binary Relevance + Logistic Regression



## Build Classifier

In [37]:
from skmultilearn.problem_transform import BinaryRelevance
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, f1_score, roc_auc_score

# solver='saga' 7it 3andna large  feature spaces
clf = BinaryRelevance(
    classifier=LogisticRegression(
        class_weight='balanced',
        solver='saga',
        max_iter=300
    ),
    #lwla Convert features to dense NumPy array
    #tania Convert labels to dense NumPy array
    require_dense=[True, True]
)

print('Classifier:')
print(clf)

Classifier:
BinaryRelevance(classifier=LogisticRegression(class_weight='balanced',
                                              max_iter=300, solver='saga'),
                require_dense=[True, True])


## Hyperparameter Tuning with GridSearchCV

GridSearchCV finds the best regularization strength C by training BR+LR with each candidate C value and evaluating via 3-fold cross-validation using macro-F1.


In [38]:
scorer = make_scorer(f1_score, average='macro')

param_grid = {'classifier__C': [0.1]}

grid_search = GridSearchCV(
    clf,
    param_grid,
    scoring=scorer,
    cv=3,
    n_jobs=-1,
    verbose=1
)

print('Fitting GridSearchCV...')
grid_search.fit(X_train, y_train)

print(f'\nBest C         : {grid_search.best_params_["classifier__C"]}')
print(f'Best CV macro-F1 : {grid_search.best_score_:.3f}')

Fitting GridSearchCV...
Fitting 3 folds for each of 1 candidates, totalling 3 fits


C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarnin


Best C         : 0.1
Best CV macro-F1 : 0.457


C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


## Predict on Test Set

scikit-multilearn returns sparse matrices by default so I  converted it  to dense numpy arrays for sklearn metrics.

In [39]:
best_clf = grid_search.best_estimator_

# Binary predictions (0/1) for F1
y_pred = np.array(best_clf.predict(X_test).todense())

# Probability scores for AUC
y_prob = np.array(best_clf.predict_proba(X_test).todense())

print(f'y_pred shape : {y_pred.shape}')
print(f'y_prob shape : {y_prob.shape}')

y_pred shape : (1022, 12)
y_prob shape : (1022, 12)


## Evaluate per Label

In [40]:
from sklearn.metrics import (
    f1_score, roc_auc_score, average_precision_score,
    balanced_accuracy_score, matthews_corrcoef,
    recall_score, confusion_matrix
)

rows = []

print(f'{"Label":<20} {"Bal.Acc":>8} {"MCC":>8} {"F1":>8} {"ROC AUC":>8} {"PR AUC":>8} {"Sensitivity":>12} {"Specificity":>12}')
print('-' * 96)

for i, label in enumerate(LABEL_COLS):
    yt = y_test[:, i]
    yp = y_pred[:, i]
    ypr = y_prob[:, i]

    bal_acc  = balanced_accuracy_score(yt, yp)
    mcc      = matthews_corrcoef(yt, yp)
    f1       = f1_score(yt, yp, zero_division=0)
    
    try:    roc_auc = roc_auc_score(yt, ypr)
    except: roc_auc = float('nan')
    
    try:    pr_auc = average_precision_score(yt, ypr)
    except: pr_auc = float('nan')

    sensitivity = recall_score(yt, yp, zero_division=0)  
    
    tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0,1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else float('nan') 

    rows.append({
        'Label'       : label,
        'Bal.Acc'     : round(bal_acc, 3),
        'MCC'         : round(mcc, 3),
        'F1'          : round(f1, 3),
        'ROC_AUC'     : round(roc_auc, 3),
        'PR_AUC'      : round(pr_auc, 3),
        'Sensitivity' : round(sensitivity, 3),
        'Specificity' : round(specificity, 3),
    })

    print(f'{label:<20} {bal_acc:>8.3f} {mcc:>8.3f} {f1:>8.3f} {roc_auc:>8.3f} {pr_auc:>8.3f} {sensitivity:>12.3f} {specificity:>12.3f}')

print('-' * 96)

# Macro averages
results = pd.DataFrame(rows)
macro = results.drop(columns='Label').mean()
print(f'{"MACRO AVG":<20} {macro["Bal.Acc"]:>8.3f} {macro["MCC"]:>8.3f} {macro["F1"]:>8.3f} {macro["ROC_AUC"]:>8.3f} {macro["PR_AUC"]:>8.3f} {macro["Sensitivity"]:>12.3f} {macro["Specificity"]:>12.3f}')

Label                 Bal.Acc      MCC       F1  ROC AUC   PR AUC  Sensitivity  Specificity
------------------------------------------------------------------------------------------------
floral                  0.793    0.441    0.510    0.873    0.545        0.814        0.772
fruity                  0.725    0.409    0.588    0.796    0.566        0.722        0.728
sweet                   0.722    0.345    0.470    0.814    0.521        0.781        0.663
woody                   0.750    0.366    0.442    0.808    0.394        0.733        0.768
green                   0.691    0.352    0.577    0.748    0.524        0.726        0.656
spicy                   0.754    0.380    0.471    0.831    0.445        0.812        0.696
animal_musk             0.717    0.334    0.442    0.764    0.459        0.688        0.747
earthy                  0.753    0.408    0.507    0.846    0.570        0.716        0.791
citrus                  0.762    0.265    0.247    0.863    0.204        0.

## Comparison Table 

In [41]:


#  BR + LR results 
br_lr = {
    'floral'       : {'Bal.Acc':0.812,'MCC':0.546,'F1':0.656,'ROC_AUC':0.875,'PR_AUC':0.672,'Sensitivity':0.854,'Specificity':0.771},
    'fruity'       : {'Bal.Acc':0.772,'MCC':0.537,'F1':0.736,'ROC_AUC':0.852,'PR_AUC':0.804,'Sensitivity':0.776,'Specificity':0.768},
    'sweet'        : {'Bal.Acc':0.682,'MCC':0.348,'F1':0.609,'ROC_AUC':0.742,'PR_AUC':0.595,'Sensitivity':0.784,'Specificity':0.579},
    'woody'        : {'Bal.Acc':0.771,'MCC':0.454,'F1':0.564,'ROC_AUC':0.837,'PR_AUC':0.567,'Sensitivity':0.763,'Specificity':0.779},
    'green'        : {'Bal.Acc':0.713,'MCC':0.419,'F1':0.673,'ROC_AUC':0.776,'PR_AUC':0.667,'Sensitivity':0.740,'Specificity':0.686},
    'spicy'        : {'Bal.Acc':0.737,'MCC':0.380,'F1':0.507,'ROC_AUC':0.805,'PR_AUC':0.505,'Sensitivity':0.795,'Specificity':0.678},
    'animal_musk'  : {'Bal.Acc':0.703,'MCC':0.302,'F1':0.406,'ROC_AUC':0.797,'PR_AUC':0.442,'Sensitivity':0.662,'Specificity':0.743},
    'earthy'       : {'Bal.Acc':0.738,'MCC':0.395,'F1':0.515,'ROC_AUC':0.820,'PR_AUC':0.571,'Sensitivity':0.700,'Specificity':0.776},
    'citrus'       : {'Bal.Acc':0.766,'MCC':0.354,'F1':0.393,'ROC_AUC':0.851,'PR_AUC':0.390,'Sensitivity':0.735,'Specificity':0.797},
    'chemical'     : {'Bal.Acc':0.717,'MCC':0.423,'F1':0.668,'ROC_AUC':0.779,'PR_AUC':0.654,'Sensitivity':0.765,'Specificity':0.670},
    'gourmand'     : {'Bal.Acc':0.748,'MCC':0.419,'F1':0.557,'ROC_AUC':0.817,'PR_AUC':0.584,'Sensitivity':0.776,'Specificity':0.719},
    'powdery_amber': {'Bal.Acc':0.739,'MCC':0.393,'F1':0.515,'ROC_AUC':0.810,'PR_AUC':0.465,'Sensitivity':0.727,'Specificity':0.751},
}

# JadBio results 
jadbio = {
    'floral'       : {'Bal.Acc':0.750,'MCC':0.526,'F1':0.638,'ROC_AUC':0.857,'PR_AUC':0.817,'Sensitivity':0.595,'Specificity':0.905},
    'fruity'       : {'Bal.Acc':0.799,'MCC':0.600,'F1':0.778,'ROC_AUC':0.872,'PR_AUC':0.871,'Sensitivity':0.762,'Specificity':0.835},
    'sweet'        : {'Bal.Acc':0.681,'MCC':0.366,'F1':0.603,'ROC_AUC':0.753,'PR_AUC':0.749,'Sensitivity':0.587,'Specificity':0.775},
    'woody'        : {'Bal.Acc':0.691,'MCC':0.468,'F1':0.530,'ROC_AUC':0.845,'PR_AUC':0.800,'Sensitivity':0.427,'Specificity':0.955},
    'green'        : {'Bal.Acc':0.709,'MCC':0.426,'F1':0.665,'ROC_AUC':0.796,'PR_AUC':0.796,'Sensitivity':0.623,'Specificity':0.794},
    'spicy'        : {'Bal.Acc':0.629,'MCC':0.354,'F1':0.410,'ROC_AUC':0.794,'PR_AUC':0.740,'Sensitivity':0.302,'Specificity':0.956},
    'animal_musk'  : {'Bal.Acc':0.665,'MCC':0.449,'F1':0.472,'ROC_AUC':0.805,'PR_AUC':0.753,'Sensitivity':0.354,'Specificity':0.975},
    'earthy'       : {'Bal.Acc':0.683,'MCC':0.470,'F1':0.516,'ROC_AUC':0.802,'PR_AUC':0.776,'Sensitivity':0.401,'Specificity':0.964},
    'citrus'       : {'Bal.Acc':0.606,'MCC':0.370,'F1':0.334,'ROC_AUC':0.842 ,'PR_AUC':0.741,'Sensitivity':0.221,'Specificity':0.991},
    'chemical'     : {'Bal.Acc':0.728,'MCC':0.466,'F1':0.678,'ROC_AUC':0.800,'PR_AUC':0.787,'Sensitivity':0.639,'Specificity':0.817},
    'gourmand'     : {'Bal.Acc':0.671,'MCC':0.450,'F1':0.502,'ROC_AUC':0.845,'PR_AUC':0.807,'Sensitivity':0.377,'Specificity':0.964},
    'powdery_amber': {'Bal.Acc':0.628,'MCC':0.317,'F1':0.403,'ROC_AUC':0.792,'PR_AUC':0.717,'Sensitivity':0.323,'Specificity':0.933},
}

#  Build comparison table 
metrics = ['Bal.Acc','MCC','F1','ROC_AUC','PR_AUC','Sensitivity','Specificity']
rows = []

for label in LABEL_COLS:
    row = {'Label': label}
    for m in metrics:
        row[f'BR_LR_{m}']  = br_lr[label][m]
        row[f'JadBio_{m}'] = jadbio[label][m]
    rows.append(row)

results = pd.DataFrame(rows)

# Macro averages 
macro_br    = {m: round(pd.DataFrame(br_lr).T[m].mean(), 3) for m in metrics}
macro_jadbio = {m: round(pd.DataFrame({k:v for k,v in jadbio.items() if v['F1'] is not None}).T[m].astype(float).mean(), 3) for m in metrics}

macro_row = {'Label': 'MACRO AVG'}
for m in metrics:
    macro_row[f'BR_LR_{m}']  = macro_br[m]
    macro_row[f'JadBio_{m}'] = macro_jadbio[m]

results = pd.concat([results, pd.DataFrame([macro_row])], ignore_index=True)

# Display
print('BR+LR vs JadBio — Full Metric Comparison')
print()
print(f'{"Label":<20}', end='')
for m in metrics:
    print(f'  {"BR_"+m:>12} {"JDB_"+m:>12}', end='')
print()
print('-' * (20 + 28*len(metrics)))
for _, row in results.iterrows():
    print(f'{row["Label"]:<20}', end='')
    for m in metrics:
        br_val  = row[f'BR_LR_{m}']
        jdb_val = row[f'JadBio_{m}']
        br_str  = f'{br_val:.3f}'  if pd.notna(br_val)  else ' None'
        jdb_str = f'{jdb_val:.3f}' if pd.notna(jdb_val) else ' None'
        print(f'  {br_str:>12} {jdb_str:>12}', end='')
    print()

BR+LR vs JadBio — Full Metric Comparison

Label                   BR_Bal.Acc  JDB_Bal.Acc        BR_MCC      JDB_MCC         BR_F1       JDB_F1    BR_ROC_AUC  JDB_ROC_AUC     BR_PR_AUC   JDB_PR_AUC  BR_Sensitivity JDB_Sensitivity  BR_Specificity JDB_Specificity
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
floral                       0.812        0.750         0.546        0.526         0.656        0.638         0.875        0.857         0.672        0.817         0.854        0.595         0.771        0.905
fruity                       0.772        0.799         0.537        0.600         0.736        0.778         0.852        0.872         0.804        0.871         0.776        0.762         0.768        0.835
sweet                        0.682        0.681         0.348        0.366         0.609        0.603

# Multi-Model Comparison: Binary Relevance

We evaluate 5 base classifiers wrapped in Binary Relevance:
- **LR** — Logistic Regression (already tuned above, results stored in `results_lr`)
- **RF** — Random Forest
- **XGB** — XGBoost
- **SVM** — Support Vector Machine (RBF kernel)
- **KNN** — k-Nearest Neighbors

Each model is tuned with GridSearchCV (macro-F1, 3-fold CV) then evaluated on the held-out test set.

In [42]:
from sklearn.metrics import (
    f1_score, roc_auc_score, average_precision_score,
    balanced_accuracy_score, matthews_corrcoef,
    recall_score, confusion_matrix
)

def find_thresholds(y_true, y_prob, label_cols, thresholds=np.arange(0.1, 0.91, 0.05)):
    """Find per-label threshold maximizing F1 on the provided set."""
    best_thresholds = {}
    for i, label in enumerate(label_cols):
        best_t, best_f1 = 0.5, 0.0
        for t in thresholds:
            y_pred_t = (y_prob[:, i] >= t).astype(int)
            f1 = f1_score(y_true[:, i], y_pred_t, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_t = t
        best_thresholds[label] = best_t
    return best_thresholds


def evaluate(name, clf, X_tr, y_tr, X_te, y_te, label_cols):
    """Fit clf, find per-label thresholds on train, evaluate on test."""
    clf.fit(X_tr, y_tr)
    
    # Get probabilities on both sets
    y_prob_train = np.array(clf.predict_proba(X_tr).todense())
    y_prob_test  = np.array(clf.predict_proba(X_te).todense())
    
    # Find per-label thresholds on TRAIN (never touch test here)
    thresholds = find_thresholds(y_tr, y_prob_train, label_cols)
    
    rows = []
    for i, label in enumerate(label_cols):
        yt  = y_te[:, i]
        ypr = y_prob_test[:, i]
        yp  = (ypr >= thresholds[label]).astype(int)   # <-- per-label threshold
        
        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0,1]).ravel()
        try:    roc = roc_auc_score(yt, ypr)
        except: roc = float('nan')
        try:    pr = average_precision_score(yt, ypr)
        except: pr = float('nan')
        rows.append({
            'Model'       : name,
            'Label'       : label,
            'Threshold'   : round(thresholds[label], 2),
            'Bal.Acc'     : round(balanced_accuracy_score(yt, yp), 3),
            'MCC'         : round(matthews_corrcoef(yt, yp), 3),
            'F1'          : round(f1_score(yt, yp, zero_division=0), 3),
            'ROC_AUC'     : round(roc, 3),
            'PR_AUC'      : round(pr, 3),
            'Sensitivity' : round(recall_score(yt, yp, zero_division=0), 3),
            'Specificity' : round(tn / (tn + fp) if (tn + fp) > 0 else float('nan'), 3),
        })
    return pd.DataFrame(rows)


def print_results(df_res, name):
    print(f'\n=== {name} ===')
    print(f'{"Label":<20} {"Bal.Acc":>8} {"MCC":>8} {"F1":>8} {"ROC_AUC":>8} {"PR_AUC":>8} {"Sensitivity":>12} {"Specificity":>12}')
    print('-' * 96)
    for _, r in df_res.iterrows():
        print(f'{r["Label"]:<20} {r["Bal.Acc"]:>8.3f} {r["MCC"]:>8.3f} {r["F1"]:>8.3f} {r["ROC_AUC"]:>8.3f} {r["PR_AUC"]:>8.3f} {r["Sensitivity"]:>12.3f} {r["Specificity"]:>12.3f}')
    print('-' * 96)
    macro = df_res.drop(columns=['Model','Label']).mean()
    print(f'{"MACRO AVG":<20} {macro["Bal.Acc"]:>8.3f} {macro["MCC"]:>8.3f} {macro["F1"]:>8.3f} {macro["ROC_AUC"]:>8.3f} {macro["PR_AUC"]:>8.3f} {macro["Sensitivity"]:>12.3f} {macro["Specificity"]:>12.3f}')


## BR + Logistic Regression

Already tuned above. We re-evaluate using the same `evaluate()` helper for consistency.

In [43]:
from skmultilearn.problem_transform import BinaryRelevance
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer

scorer = make_scorer(f1_score, average='macro')

lr_clf = BinaryRelevance(
    classifier=LogisticRegression(class_weight='balanced', solver='saga', max_iter=300),
    require_dense=[True, True]
)
gs_lr = GridSearchCV(lr_clf, {'classifier__C': [0.01]},
                     scoring=scorer, cv=3, n_jobs=-1, verbose=1)
gs_lr.fit(X_train, y_train)
print(f'Best C : {gs_lr.best_params_["classifier__C"]}  |  CV macro-F1 : {gs_lr.best_score_:.3f}')

results_lr = evaluate('LR', gs_lr.best_estimator_, X_train, y_train, X_test, y_test, LABEL_COLS)
print_results(results_lr, 'BR + Logistic Regression')

Fitting 3 folds for each of 1 candidates, totalling 3 fits


C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarnin

Best C : 0.01  |  CV macro-F1 : 0.461


C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarnin


=== BR + Logistic Regression ===
Label                 Bal.Acc      MCC       F1  ROC_AUC   PR_AUC  Sensitivity  Specificity
------------------------------------------------------------------------------------------------
floral                  0.772    0.451    0.532    0.867    0.525        0.697        0.847
fruity                  0.709    0.389    0.571    0.791    0.554        0.657        0.761
sweet                   0.721    0.377    0.501    0.807    0.501        0.645        0.797
woody                   0.696    0.342    0.435    0.815    0.392        0.519        0.872
green                   0.674    0.320    0.562    0.745    0.523        0.784        0.563
spicy                   0.732    0.400    0.503    0.832    0.450        0.625        0.840
animal_musk             0.704    0.397    0.491    0.771    0.462        0.506        0.901
earthy                  0.761    0.507    0.588    0.844    0.568        0.611        0.912
citrus                  0.690    0.270   

## BR + Random Forest

In [44]:
from sklearn.ensemble import RandomForestClassifier

rf_clf = BinaryRelevance(
    classifier=RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
    require_dense=[True, True]
)
gs_rf = GridSearchCV(
    rf_clf,
    {'classifier__n_estimators': [300], 'classifier__max_features': ['sqrt']},
    scoring=scorer, cv=3, n_jobs=-1, verbose=1
)
gs_rf.fit(X_train, y_train)
print(f'Best params : {gs_rf.best_params_}  |  CV macro-F1 : {gs_rf.best_score_:.3f}')

results_rf = evaluate('RF', gs_rf.best_estimator_, X_train, y_train, X_test, y_test, LABEL_COLS)
print_results(results_rf, 'BR + Random Forest')

Fitting 3 folds for each of 1 candidates, totalling 3 fits
Best params : {'classifier__max_features': 'sqrt', 'classifier__n_estimators': 300}  |  CV macro-F1 : 0.340

=== BR + Random Forest ===
Label                 Bal.Acc      MCC       F1  ROC_AUC   PR_AUC  Sensitivity  Specificity
------------------------------------------------------------------------------------------------
floral                  0.715    0.421    0.505    0.873    0.548        0.517        0.912
fruity                  0.741    0.508    0.629    0.836    0.701        0.585        0.898
sweet                   0.727    0.450    0.550    0.823    0.600        0.557        0.897
woody                   0.641    0.340    0.398    0.829    0.432        0.328        0.953
green                   0.669    0.329    0.543    0.744    0.548        0.574        0.764
spicy                   0.720    0.441    0.528    0.836    0.538        0.525        0.914
animal_musk             0.692    0.393    0.482    0.796    0.48

## BR + XGBoost

In [45]:
from xgboost import XGBClassifier

xgb_clf = BinaryRelevance(
    classifier=XGBClassifier(eval_metric='logloss', random_state=42, n_jobs=-1),
    require_dense=[True, True]
)
gs_xgb = GridSearchCV(
    xgb_clf,
    {'classifier__n_estimators': [300], 'classifier__max_depth': [6]},
    scoring=scorer, cv=3, n_jobs=-1, verbose=1
)
gs_xgb.fit(X_train, y_train)
print(f'Best params : {gs_xgb.best_params_}  |  CV macro-F1 : {gs_xgb.best_score_:.3f}')

results_xgb = evaluate('XGB', gs_xgb.best_estimator_, X_train, y_train, X_test, y_test, LABEL_COLS)
print_results(results_xgb, 'BR + XGBoost')

Fitting 3 folds for each of 1 candidates, totalling 3 fits
Best params : {'classifier__max_depth': 6, 'classifier__n_estimators': 300}  |  CV macro-F1 : 0.397

=== BR + XGBoost ===
Label                 Bal.Acc      MCC       F1  ROC_AUC   PR_AUC  Sensitivity  Specificity
------------------------------------------------------------------------------------------------
floral                  0.775    0.504    0.578    0.868    0.548        0.648        0.902
fruity                  0.755    0.477    0.631    0.825    0.697        0.715        0.795
sweet                   0.749    0.447    0.555    0.830    0.611        0.650        0.849
woody                   0.680    0.329    0.422    0.802    0.389        0.473        0.887
green                   0.651    0.280    0.529    0.720    0.513        0.635        0.666
spicy                   0.704    0.359    0.470    0.804    0.476        0.562        0.846
animal_musk             0.694    0.365    0.467    0.760    0.461        0.506

## BR + SVM

SVM requires `probability=True` for `predict_proba` (needed for ROC AUC / PR AUC). This makes fitting slower.

In [46]:
from sklearn.svm import SVC

svm_clf = BinaryRelevance(
    classifier=SVC(class_weight='balanced', kernel='rbf', probability=True, random_state=42),
    require_dense=[True, True]
)
gs_svm = GridSearchCV(
    svm_clf,
    {'classifier__C': [1]},
    scoring=scorer, cv=3, n_jobs=-1, verbose=1
)
gs_svm.fit(X_train, y_train)
print(f'Best C : {gs_svm.best_params_["classifier__C"]}  |  CV macro-F1 : {gs_svm.best_score_:.3f}')

results_svm = evaluate('SVM', gs_svm.best_estimator_, X_train, y_train, X_test, y_test, LABEL_COLS)
print_results(results_svm, 'BR + SVM')

Fitting 3 folds for each of 1 candidates, totalling 3 fits
Best C : 1  |  CV macro-F1 : 0.469

=== BR + SVM ===
Label                 Bal.Acc      MCC       F1  ROC_AUC   PR_AUC  Sensitivity  Specificity
------------------------------------------------------------------------------------------------
floral                  0.729    0.463    0.538    0.880    0.552        0.531        0.927
fruity                  0.730    0.483    0.611    0.820    0.630        0.570        0.890
sweet                   0.748    0.449    0.556    0.830    0.558        0.639        0.856
woody                   0.605    0.265    0.325    0.818    0.384        0.260        0.951
green                   0.686    0.357    0.566    0.759    0.556        0.616        0.756
spicy                   0.687    0.397    0.483    0.842    0.506        0.450        0.923
animal_musk             0.656    0.386    0.444    0.782    0.506        0.357        0.955
earthy                  0.660    0.410    0.458    0.84

## BR + k-Nearest Neighbors

In [47]:
from sklearn.neighbors import KNeighborsClassifier

knn_clf = BinaryRelevance(
    classifier=KNeighborsClassifier(n_jobs=-1),
    require_dense=[True, True]
)
gs_knn = GridSearchCV(
    knn_clf,
    {'classifier__n_neighbors': [5], 'classifier__metric': ['cosine']},
    scoring=scorer, cv=3, n_jobs=-1, verbose=1
)
gs_knn.fit(X_train, y_train)
print(f'Best params : {gs_knn.best_params_}  |  CV macro-F1 : {gs_knn.best_score_:.3f}')

results_knn = evaluate('KNN', gs_knn.best_estimator_, X_train, y_train, X_test, y_test, LABEL_COLS)
print_results(results_knn, 'BR + KNN')

Fitting 3 folds for each of 1 candidates, totalling 3 fits
Best params : {'classifier__metric': 'cosine', 'classifier__n_neighbors': 5}  |  CV macro-F1 : 0.400

=== BR + KNN ===
Label                 Bal.Acc      MCC       F1  ROC_AUC   PR_AUC  Sensitivity  Specificity
------------------------------------------------------------------------------------------------
floral                  0.748    0.425    0.513    0.833    0.475        0.634        0.861
fruity                  0.732    0.421    0.596    0.809    0.605        0.744        0.721
sweet                   0.723    0.376    0.500    0.791    0.457        0.661        0.785
woody                   0.686    0.319    0.416    0.750    0.382        0.511        0.861
green                   0.651    0.280    0.532    0.703    0.481        0.658        0.645
spicy                   0.671    0.298    0.422    0.740    0.381        0.512        0.829
animal_musk             0.681    0.316    0.431    0.746    0.361        0.519   

## Summary: Macro-Averaged Metrics Across All Models

Aggregate per-model macro averages for a side-by-side comparison.

In [48]:
all_results = pd.concat([results_lr, results_rf, results_xgb, results_svm, results_knn], ignore_index=True)

metric_cols = ['Bal.Acc', 'MCC', 'F1', 'ROC_AUC', 'PR_AUC', 'Sensitivity', 'Specificity']
summary = (
    all_results.groupby('Model')[metric_cols]
    .mean()
    .round(3)
    .loc[['LR', 'RF', 'XGB', 'SVM', 'KNN']]  # fix display order
)

print('Macro-averaged test metrics per model')
print(summary.to_string())

Macro-averaged test metrics per model
       Bal.Acc    MCC     F1  ROC_AUC  PR_AUC  Sensitivity  Specificity
Model                                                                  
LR       0.721  0.395  0.525    0.812   0.496        0.637        0.804
RF       0.693  0.411  0.516    0.827   0.560        0.500        0.887
XGB      0.712  0.397  0.527    0.807   0.539        0.603        0.822
SVM      0.687  0.399  0.505    0.822   0.530        0.493        0.881
KNN      0.697  0.353  0.491    0.771   0.452        0.591        0.803


In [49]:
MORGAN_RADIUS = 1

# Add radius column to results
all_results['morgan_radius'] = MORGAN_RADIUS

# Save per-label results
all_results.to_csv(f'baseline_results_radius_{MORGAN_RADIUS}.csv', index=False)

# Save macro summary
metric_cols = ['Bal.Acc', 'F1', 'ROC_AUC', 'PR_AUC', 'Sensitivity', 'Specificity']
summary = (
    all_results.groupby('Model')[metric_cols]
    .mean()
    .round(3)
    .reset_index()
)
summary['morgan_radius'] = MORGAN_RADIUS
summary.to_csv(f'baseline_summary_radius_{MORGAN_RADIUS}.csv', index=False)

print(f'Saved results for radius={MORGAN_RADIUS}')

Saved results for radius=1
